# Neural Network (NN1) Feature Set: MALL

**Purspose:** This notebook implements the NN1 neural network architecture from Christensen, 
Siggaard, and Veliyev (2023) on the extended MALL feature set for the EURO STOXX 
50 index over the period 1999–2020. We use the same NN1 architecture, 
hyperparameters, and monthly rolling window estimation scheme as in our MHAR 
implementation, with the only difference being the expanded feature set of 20 
predictors. The primary objective is to establish whether the inclusion of 
additional macroeconomic and market predictors improves forecast accuracy relative 
to the NN1 estimated on the parsimonious MHAR feature set, consistent with the 
findings of Christensen et al. (2023) who document that ML models are particularly 
adept at extracting incremental information from a rich information set.

## 1. Imports 

We import the necessary libraries for data handling, numerical computation, and 
neural network estimation. We use TensorFlow and Keras for the network 
implementation. Random seeds are set for both NumPy and TensorFlow to ensure 
reproducibility of results across runs.

In [4]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

TensorFlow version: 2.16.2
Keras version: 3.10.0


## 2. Data Loading 

We load the MALL dataset which contains all predictor variables and the target 
variable $RV_{t+1}$ for the EURO STOXX 50 index over the period 1999–2020. 
We select the 20 predictors from the reduced MALL feature set, excluding redundant 
variables such as log transformations and level variables where the return 
counterpart is more appropriate for volatility forecasting. This reduces the 
feature set from 29 to 20 variables while retaining the most informative predictors, 
and ensures computational feasibility for the rolling window estimation.

In [7]:
# Load MALL dataset
file_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data/MALL.csv"
MALL = pd.read_csv(file_path, index_col='Date', parse_dates=True)

print(f"Full dataset shape: {MALL.shape}")
print(f"Date range: {MALL.index[0].date()} to {MALL.index[-1].date()}")
print(f"NaN values: {MALL.isna().sum().sum()}")

# Define reduced MALL feature set — 20 predictors
# We exclude log transformations and redundant level variables
MALL_features = [
    # Core HAR variables
    "RVD", "RVW", "RVM",
    # Implied volatility
    "VSTOXX",
    # Policy uncertainty (lagged to avoid look-ahead bias)
    "EPU_1m_lag",
    # Exchange rate return
    "EURUSD_ret",
    # Systemic stress
    "CISS",
    # Oil price return
    "BRENT_ret",
    # Asian market
    "NIKKEI_ret", "NIKKEI_sq",
    # US market
    "SP500_ret_lag", "SP500_sq_lag",
    # Momentum
    "M1W", "M1M"
]

print(f"\nSelected MALL features ({len(MALL_features)}):")
for i, f in enumerate(MALL_features, 1):
    print(f"  {i:2d}. {f}")

# Verify all features exist in MALL
missing = [f for f in MALL_features if f not in MALL.columns]
print(f"\nMissing features: {missing if missing else 'None'}")

Full dataset shape: (5488, 30)
Date range: 1999-01-05 to 2020-08-05
NaN values: 0

Selected MALL features (14):
   1. RVD
   2. RVW
   3. RVM
   4. VSTOXX
   5. EPU_1m_lag
   6. EURUSD_ret
   7. CISS
   8. BRENT_ret
   9. NIKKEI_ret
  10. NIKKEI_sq
  11. SP500_ret_lag
  12. SP500_sq_lag
  13. M1W
  14. M1M

Missing features: None


## 3. Data Split
### 3.1 Train / Validation / Test Split (70 / 10 / 20)

Following Christensen et al. (2023), we split the sample into a training set 
(70%), validation set (10%), and test set (20%). The split is identical to that 
used in our MHAR implementation, ensuring a direct apples-to-apples comparison 
between the two feature sets. The training set is used to estimate the network 
weights, the validation set is used for early stopping and ensemble selection, 
and the test set is reserved exclusively for out-of-sample evaluation.

In [9]:
# Train / Validation / Test split (70 / 10 / 20)
n = len(MALL)
n_train = int(np.floor(0.70 * n))
n_val   = int(np.floor(0.10 * n))
n_test  = n - n_train - n_val

print(f"Total observations: {n}")
print(f"\nTrain:      {MALL.index[0].date()} to {MALL.index[n_train-1].date()} ({n_train} obs, {n_train/n*100:.1f}%)")
print(f"Validation: {MALL.index[n_train].date()} to {MALL.index[n_train+n_val-1].date()} ({n_val} obs, {n_val/n*100:.1f}%)")
print(f"Test:       {MALL.index[n_train+n_val].date()} to {MALL.index[-1].date()} ({n_test} obs, {n_test/n*100:.1f}%)")

Total observations: 5488

Train:      1999-01-05 to 2014-02-25 (3841 obs, 70.0%)
Validation: 2014-02-26 to 2016-04-19 (548 obs, 10.0%)
Test:       2016-04-20 to 2020-08-05 (1099 obs, 20.0%)


## 4. Feature Set Construction and Standardization

We construct the MALL feature set using the 20 selected predictors and standardize 
all inputs and the target variable using the training set mean and standard deviation. 
Following Christensen et al. (2023), standardization parameters are computed 
exclusively from the training set and applied consistently across the full dataset, 
ensuring no look-ahead bias is introduced. We scale the full dataset upfront using 
these fixed training set parameters, which is required for the rolling window 
estimation where the window moves forward one day at a time across the test period.

In [11]:
# Define target and MALL feature set
y      = MALL["RV_target"].values
X_MALL = MALL[MALL_features].values

# Split into train, validation and test sets
X_train = X_MALL[:n_train]
X_val   = X_MALL[n_train:n_train + n_val]
X_test  = X_MALL[n_train + n_val:]

y_train = y[:n_train]
y_val   = y[n_train:n_train + n_val]
y_test  = y[n_train + n_val:]

# Compute standardization parameters from training set only
X_mean = X_train.mean(axis=0)
X_std  = X_train.std(axis=0)
y_mean = y_train.mean()
y_std  = y_train.std()

# Scale full dataset using training set parameters
X_scaled = (X_MALL - X_mean) / X_std
y_scaled = (y      - y_mean) / y_std

# Also store scaled train/val/test splits for verification
X_train_scaled = X_scaled[:n_train]
X_val_scaled   = X_scaled[n_train:n_train + n_val]
X_test_scaled  = X_scaled[n_train + n_val:]

y_train_scaled = y_scaled[:n_train]
y_val_scaled   = y_scaled[n_train:n_train + n_val]
y_test_scaled  = y_scaled[n_train + n_val:]

print(f"Feature set: {len(MALL_features)} predictors")
print(f"\nFull dataset:   X={X_scaled.shape}, y={y_scaled.shape}")
print(f"Training set:   X={X_train_scaled.shape}, y={y_train_scaled.shape}")
print(f"Validation set: X={X_val_scaled.shape},   y={y_val_scaled.shape}")
print(f"Test set:       X={X_test_scaled.shape},  y={y_test_scaled.shape}")
print(f"\nStandardization parameters from training set:")
print(f"  y mean: {y_mean:.6e}")
print(f"  y std:  {y_std:.6e}")
print(f"\nVerification — training set after standardization:")
print(f"  X mean (should be ~0): {X_train_scaled.mean(axis=0).round(4)}")
print(f"  X std  (should be ~1): {X_train_scaled.std(axis=0).round(4)}")
print(f"  y mean (should be ~0): {y_train_scaled.mean():.6f}")
print(f"  y std  (should be ~1): {y_train_scaled.std():.6f}")

Feature set: 14 predictors

Full dataset:   X=(5488, 14), y=(5488,)
Training set:   X=(3841, 14), y=(3841,)
Validation set: X=(548, 14),   y=(548,)
Test set:       X=(1099, 14),  y=(1099,)

Standardization parameters from training set:
  y mean: 1.633669e-04
  y std:  2.562095e-04

Verification — training set after standardization:
  X mean (should be ~0): [ 0.  0. -0. -0. -0. -0. -0.  0.  0. -0.  0.  0. -0.  0.]
  X std  (should be ~1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
  y mean (should be ~0): 0.000000
  y std  (should be ~1): 1.000000


## 5. NN1 Architecture

We implement the same NN1 architecture as in our MHAR implementation, following 
Christensen et al. (2023). The only difference is the input dimension, which 
increases from $J = 3$ for the MHAR feature set to $J = 20$ for the MALL feature 
set. This increases the number of trainable parameters in the first hidden layer 
from 8 to 42, bringing the total parameter count from 11 to 45. Despite this 
increase, NN1 remains a highly parsimonious model relative to the size of the 
training set, and all other architectural choices — Leaky ReLU activation, Glorot 
normal initialization, dropout rate of 0.2, Adam optimizer with learning rate 
0.001, and linear output activation — are kept identical to the MHAR implementation.

In [13]:
def build_NN1(input_dim, seed=None):
    """
    NN1 architecture following Christensen et al. (2023):

    Architecture:
        - Input layer:  input_dim neurons
        - Hidden layer: 2 neurons, Leaky ReLU (c=0.01), Glorot normal init
        - Dropout:      rate=0.2 (keep rate=0.8 as in paper)
        - Output layer: 1 neuron, linear activation

    Optimizer:   Adam, learning rate=0.001
    Loss:        Mean Squared Error
    """
    if seed is not None:
        tf.random.set_seed(seed)

    initializer = keras.initializers.GlorotNormal(seed=seed)

    model = keras.Sequential([
        keras.Input(shape=(input_dim,)),
        layers.Dense(
            2,
            kernel_initializer=initializer,
            bias_initializer="zeros"
        ),
        layers.LeakyReLU(negative_slope=0.01),
        layers.Dropout(rate=0.2),
        layers.Dense(
            1,
            kernel_initializer=initializer,
            bias_initializer="zeros",
            activation="linear"
        )
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mse"
    )

    return model

# Print model summary for MALL input dimension
test_model = build_NN1(input_dim=20, seed=0)
test_model.summary()
print(f"\nVerification of parameter count:")
print(f"  Input -> Hidden: 20 inputs x 2 neurons + 2 bias = 42 parameters")
print(f"  Hidden -> Output: 2 inputs x 1 neuron + 1 bias  =  3 parameters")
print(f"  Total:                                           = 45 parameters")

2026-03-22 09:15:09.761069: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-03-22 09:15:09.761090: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-03-22 09:15:09.761098: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-03-22 09:15:09.761112: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-22 09:15:09.761122: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 2)              │            42 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 2)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45 (180.00 B)

 Trainable params: 45 (180.00 B)

 Non-trainable params: 0 (0.00 B)


Verification of parameter count:
  Input -> Hidden: 20 inputs x 2 neurons + 2 bias = 42 parameters
  Hidden -> Output: 2 inputs x 1 neuron + 1 bias  =  3 parameters
  Total:                                           = 45 parameters


## 6. Hyperparameters

We adopt the same hyperparameters as in our MHAR implementation, ensuring a 
direct comparison between the two feature sets. The ensemble size, maximum 
epochs, early stopping patience, re-estimation frequency, learning rate, 
dropout rate, batch size, and weight initializer are all kept identical. 
The only difference is the input dimension, which increases from $J = 3$ 
to $J = 20$, marginally increasing the computational cost per refit due to 
the larger number of parameters in the first hidden layer.

In [15]:
# Hyperparameters — identical to MHAR implementation
n_ensemble = 10   # reduced from 100 (paper) to 10
n_best     = 3    # reduced from 10 (paper) to 3
max_epochs = 200  # reduced from 500 (paper) to 200
patience   = 50   # early stopping patience
refit_freq = 22   # re-estimate every 22 trading days

print("Hyperparameter summary:")
print(f"  Ensemble size:        {n_ensemble} networks (paper: 100)")
print(f"  Networks selected:    {n_best} best (paper: 10 out of 100)")
print(f"  Max epochs:           {max_epochs} (paper: 500)")
print(f"  Early stopping:       patience={patience} (paper: 100)")
print(f"  Re-estimation freq:   every {refit_freq} trading days")
print(f"  Learning rate:        0.001 (identical to paper)")
print(f"  Dropout keep rate:    0.8 (identical to paper)")
print(f"  Batch size:           full batch (identical to paper)")
print(f"  Weight initializer:   Glorot normal (identical to paper)")
print(f"  Activation:           Leaky ReLU c=0.01 (identical to paper)")
print(f"  Input dimension:      {len(MALL_features)} (MHAR: 3)")
print(f"  Total parameters:     45 (MHAR: 11)")

# Estimate computational cost
n_refits       = int(np.ceil(n_test / refit_freq))
time_per_refit = 1.5   # slightly higher than MHAR due to larger input dim
total_time     = n_refits * time_per_refit

print(f"\nComputational cost estimate:")
print(f"  Number of refits:     {n_refits}")
print(f"  Time per refit:       ~{time_per_refit} minutes")
print(f"  Total estimated time: ~{total_time:.0f} minutes ({total_time/60:.1f} hours)")

Hyperparameter summary:
  Ensemble size:        10 networks (paper: 100)
  Networks selected:    3 best (paper: 10 out of 100)
  Max epochs:           200 (paper: 500)
  Early stopping:       patience=50 (paper: 100)
  Re-estimation freq:   every 22 trading days
  Learning rate:        0.001 (identical to paper)
  Dropout keep rate:    0.8 (identical to paper)
  Batch size:           full batch (identical to paper)
  Weight initializer:   Glorot normal (identical to paper)
  Activation:           Leaky ReLU c=0.01 (identical to paper)
  Input dimension:      14 (MHAR: 3)
  Total parameters:     45 (MHAR: 11)

Computational cost estimate:
  Number of refits:     50
  Time per refit:       ~1.5 minutes
  Total estimated time: ~75 minutes (1.2 hours)


## 7. Monthly Rolling Window Estimation and Forecasting

We train 10 independent NN1 networks with different random seeds on a rolling 
window of fixed length equal to the combined training and validation set (4,389 
observations). The ensemble is re-estimated every 22 trading days, with the 
validation portion of each window used for early stopping and ensemble selection. 
We select the 3 best networks based on validation MSE and average their forecasts. 
Between re-estimation dates we use the most recently estimated ensemble to generate 
daily forecasts. All forecasts are back-transformed to the original scale by 
reversing the target standardization. We apply the insanity filter of Bollerslev, 
Patton, and Quaedvlieg (2016), replacing any negative forecast with the minimum 
realized variance in the current estimation window.

In [ ]:
# Rolling window size = train + validation
window_size = n_train + n_val

forecasts_NN1_MALL = []
dates_test         = []
best_models        = None
last_refit         = -refit_freq  # force refit on first iteration

start_time = time.time()

for i in range(n_test):

    # Re-estimate ensemble every refit_freq days
    if i - last_refit >= refit_freq:

        # Extract rolling window of scaled data
        X_window = X_scaled[i : i + window_size]
        y_window = y_scaled[i : i + window_size]

        # Split window into train and validation
        X_win_train = X_window[:-n_val]
        X_win_val   = X_window[-n_val:]
        y_win_train = y_window[:-n_val]
        y_win_val   = y_window[-n_val:]

        # Train ensemble of n_ensemble networks
        val_losses_window = []
        models_window     = []

        for j in range(n_ensemble):

            early_stopping = EarlyStopping(
                monitor="val_loss",
                patience=patience,
                restore_best_weights=True,
                verbose=0
            )

            model = build_NN1(input_dim=len(MALL_features), seed=j)
            model.fit(
                X_win_train, y_win_train,
                validation_data=(X_win_val, y_win_val),
                epochs=max_epochs,
                batch_size=len(X_win_train),
                callbacks=[early_stopping],
                verbose=0
            )
            val_loss = model.evaluate(X_win_val, y_win_val, verbose=0)
            val_losses_window.append(val_loss)
            models_window.append(model)

        # Select n_best networks based on validation MSE
        top_idx     = np.argsort(val_losses_window)[:n_best]
        best_models = [models_window[j] for j in top_idx]
        last_refit  = i

        # Progress update
        elapsed   = (time.time() - start_time) / 60
        refit_num = i // refit_freq + 1
        print(f"  Refit {refit_num:3d}/{n_refits} | "
              f"Test day {i:4d}/{n_test} | "
              f"Date: {MALL.index[i + window_size].date()} | "
              f"Best val MSE: {min(val_losses_window):.4f} | "
              f"Elapsed: {elapsed:.1f} min")

    # Generate ensemble forecast for current test day
    X_forecast = X_scaled[i + window_size].reshape(1, -1)

    preds_scaled = np.array([
        model.predict(X_forecast, verbose=0).flatten()[0]
        for model in best_models
    ])

    # Average ensemble predictions and back-transform to original scale
    forecast = preds_scaled.mean() * y_std + y_mean

    # Insanity filter: replace negative forecasts with minimum in-sample RV
    min_rv   = y[i : i + window_size].min()
    forecast = max(forecast, min_rv)

    forecasts_NN1_MALL.append(forecast)
    dates_test.append(MALL.index[i + window_size])

# Total time
total_elapsed = (time.time() - start_time) / 60
print(f"\nTotal estimation time: {total_elapsed:.1f} minutes")
print(f"Forecasts generated: {len(forecasts_NN1_MALL)}")
print(f"Forecast period: {dates_test[0].date()} to {dates_test[-1].date()}")

2026-03-22 09:15:10.153714: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


  Refit   1/50 | Test day    0/1099 | Date: 2016-04-20 | Best val MSE: 0.2841 | Elapsed: 1.2 min
  Refit   2/50 | Test day   22/1099 | Date: 2016-05-20 | Best val MSE: 0.2828 | Elapsed: 2.5 min
  Refit   3/50 | Test day   44/1099 | Date: 2016-06-21 | Best val MSE: 0.2845 | Elapsed: 3.8 min
  Refit   4/50 | Test day   66/1099 | Date: 2016-07-22 | Best val MSE: 0.4344 | Elapsed: 5.1 min
  Refit   5/50 | Test day   88/1099 | Date: 2016-08-23 | Best val MSE: 0.4409 | Elapsed: 6.4 min
  Refit   6/50 | Test day  110/1099 | Date: 2016-09-22 | Best val MSE: 0.4417 | Elapsed: 7.7 min
  Refit   7/50 | Test day  132/1099 | Date: 2016-10-24 | Best val MSE: 0.4457 | Elapsed: 9.0 min
  Refit   8/50 | Test day  154/1099 | Date: 2016-11-23 | Best val MSE: 0.4564 | Elapsed: 10.4 min
  Refit   9/50 | Test day  176/1099 | Date: 2016-12-23 | Best val MSE: 0.4492 | Elapsed: 11.7 min
  Refit  10/50 | Test day  198/1099 | Date: 2017-01-25 | Best val MSE: 0.4477 | Elapsed: 13.1 min
  Refit  11/50 | Test day  